# 07 — Beating persistence

The bar from `04_baseline_forecasting`: naive persistence ("next month = this month") scores MAE 153 / RMSE 330 on the FY 2024-25 validation year, and no linear model beats it. This notebook adds the structure persistence ignores — seasonality, recent momentum, backlog age, bed occupancy — using gradient-boosted trees.

Two design points, both forced by the data:

1. **The model predicts the one-month *change*, not the level.** Trees can only predict values inside the range they were trained on, and the waiting list keeps reaching new highs — a level model trained on 2018-2024 physically cannot output a 2024-25 record. The level variant is fitted anyway below, as evidence.
2. **Same cohort, same splits, same validation rows as 04** (asserted), so every number here is directly comparable to the MAE-153 bar. The sealed test year FY 2025-26 stays untouched.

In [1]:
import pandas as pd                          # tables
import numpy as np                           # array arithmetic
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
# scikit-learn's gradient-boosted trees: many small decision trees fitted in sequence,
# each correcting the previous ones' errors. The Hist variant accepts missing values
# natively, so occupancy gaps need no invented fill-in value.
from sklearn.inspection import permutation_importance
# measures feature importance by shuffling one column and watching the error grow

PROC = Path("../data/processed")
panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])

TEST_START = pd.Timestamp("2025-04-01")      # sealed test year - never read below (same as 04)
VAL_START = pd.Timestamp("2024-04-01")       # validation year FY 2024-25 (same as 04)
COVID = pd.Timestamp("2020-03-01")           # regime break: first lockdown month
SEED = 42                                    # the one declared seed, used by every model here

dev = panel[panel["month"] < TEST_START]     # development window, Apr 2018 - Mar 2025
print("dev rows:", len(dev), "| sealed rows kept out:", int((panel["month"] >= TEST_START).sum()))

dev rows: 300193 | sealed rows kept out: 43248


## Features: what persistence throws away

Persistence uses only the current level. Each feature below is information that already exists at forecast time and could explain next month's *change*: the calendar month (winter vs summer), how the list moved over the last three months (momentum), how old the backlog is (share past 18 / 52 weeks), and how full the trust's beds are (occupancy — provider-level, NaN for the few unmatched months).

In [2]:
# Goal: the 04 cohort (C_110 at stable public providers) with the one-month CHANGE
# as target, plus momentum and seasonality features.
df = dev[(dev["tfunc_code"] == "C_110") & dev["fl_client"] & dev["is_public"]].copy()
df = df.sort_values(["provider_code", "month"])

g = df.groupby("provider_code")["total_waiting"]
df["target_delta"] = g.shift(-1) - df["total_waiting"]   # next month's list minus this month's
for k in (1, 2, 3):                                      # momentum over the last 1/2/3 months:
    df[f"delta_{k}"] = df["total_waiting"] - g.shift(k)  # shift(k) = same provider, k rows earlier
df["month_of_year"] = df["month"].dt.month               # 1-12, lets the trees learn seasonality

# shift() counts reported months, not calendar months - measure how often that differs
gappy = (df.groupby("provider_code")["month"].diff() > pd.Timedelta(days=45)).mean()
print(f"row-to-row steps spanning more than one calendar month: {gappy:.2%} of rows")

df = df.dropna(subset=["target_delta"])
# drops only each provider's final dev month (nothing to predict). Momentum is NaN in a
# provider's first months (and after a reporting gap re-entry); those rows stay - the
# trees accept missing values - so the validation rows remain exactly 04's.

FEATS = ["total_waiting", "pct_gt18", "pct_gt52",
         "delta_1", "delta_2", "delta_3", "month_of_year", "occupancy_ga"]
print(df["provider_code"].nunique(), "providers,", len(df), "provider-months")

row-to-row steps spanning more than one calendar month: 0.10% of rows
130 providers, 10431 provider-months


In [3]:
# Goal: the same two training regimes and validation year as 04.
train_full = df[df["month"] < VAL_START]                              # Apr 2018 - Mar 2024
train_post = df[(df["month"] >= COVID) & (df["month"] < VAL_START)]   # Mar 2020 - Mar 2024
val = df[df["month"] >= VAL_START]                                    # FY 2024-25

assert val["month"].max() < TEST_START       # a sealed-year row leaking in would trip this
assert len(val) == 1407, len(val)            # 04 scored exactly 1,407 rows; a val row lost to
                                             # the momentum dropna would break comparability
truth = (val["total_waiting"] + val["target_delta"]).to_numpy()   # next month's actual list

naive_mae = np.abs(val["target_delta"]).mean()   # persistence's error IS the actual change
assert round(naive_mae) == 153, naive_mae        # different rows from 04 would trip this
print(f"naive on these rows: MAE {naive_mae:,.0f}   (n={len(val)}) - matches 04")
print(len(train_full), "train-full,", len(train_post), "train-post,", len(val), "validation rows")

naive on these rows: MAE 153   (n=1407) - matches 04
9024 train-full, 6185 train-post, 1407 validation rows


## Gradient boosting on the change

Forecast level = current level + predicted change. Persistence is the special case "predicted change = 0", so the model only has to predict the change better than not predicting it at all.

In [4]:
# Goal: one model per training regime, scored on the same rows and scale as 04.
def score(pred, label):
    print(f"{label}: MAE {np.abs(truth - pred).mean():,.0f}   "
          f"RMSE {np.sqrt(((truth - pred) ** 2).mean()):,.0f}   (n={len(val)})")

preds = {}
for name, tr in [("full-history", train_full), ("post-covid ", train_post)]:
    m = HistGradientBoostingRegressor(random_state=SEED)
    m.fit(tr[FEATS], tr["target_delta"])
    preds[name.strip()] = val["total_waiting"].to_numpy() + m.predict(val[FEATS])
    score(preds[name.strip()], f"gbm-delta {name}")
score(val["total_waiting"].to_numpy(), "naive        persistence")

gbm-delta full-history: MAE 155   RMSE 336   (n=1407)
gbm-delta post-covid : MAE 158   RMSE 338   (n=1407)
naive        persistence: MAE 153   RMSE 330   (n=1407)


## Why the change and not the level

The obvious first attempt — predict the raw next-month level — is fitted here to show why it was rejected: training levels top out below where FY 2024-25 sits, and a tree cannot output a value it never saw.

In [5]:
# Goal: the raw-level variant, reported as evidence for the delta design.
for name, tr in [("full-history", train_full), ("post-covid ", train_post)]:
    m = HistGradientBoostingRegressor(random_state=SEED)
    m.fit(tr[FEATS], tr["total_waiting"] + tr["target_delta"])   # next-month level as target
    score(m.predict(val[FEATS]), f"gbm-level {name}")
print("largest next-month list seen in full training:",
      int((train_full["total_waiting"] + train_full["target_delta"]).max()),
      "| largest in validation:", int(truth.max()))

gbm-level full-history: MAE 178   RMSE 372   (n=1407)
gbm-level post-covid : MAE 174   RMSE 368   (n=1407)
largest next-month list seen in full training: 18453 | largest in validation: 18758


## Transparent references: drift and seasonal drift

Before blaming the model class, two one-parameter-per-cell references. "Drift" forecasts every change as the training-mean change; "seasonal drift" forecasts each month's change as the training mean for that calendar month. If seasonality carries real signal, seasonal drift alone should already edge persistence.

In [6]:
# Goal: does the average change - overall, or per calendar month - beat forecasting zero?
for name, tr in [("full-history", train_full), ("post-covid ", train_post)]:
    score((val["total_waiting"] + tr["target_delta"].mean()).to_numpy(), f"drift          {name}")
    by_month = tr.groupby("month_of_year")["target_delta"].mean()   # e.g. mean January change
    score((val["total_waiting"] + val["month_of_year"].map(by_month)).to_numpy(),
          f"seasonal-drift {name}")

drift          full-history: MAE 156   RMSE 330   (n=1407)
seasonal-drift full-history: MAE 158   RMSE 332   (n=1407)
drift          post-covid : MAE 158   RMSE 331   (n=1407)
seasonal-drift post-covid : MAE 162   RMSE 333   (n=1407)


## What the model leaned on

Shuffling one feature column breaks its link to the target; the resulting rise in validation MAE measures how much the model used it. Near-zero means the feature could be deleted without changing the forecast.

In [7]:
# Goal: permutation importance of the post-covid delta model on the validation year.
m = HistGradientBoostingRegressor(random_state=SEED)
m.fit(train_post[FEATS], train_post["target_delta"])
imp = permutation_importance(m, val[FEATS], val["target_delta"],
                             scoring="neg_mean_absolute_error", random_state=SEED)
print(pd.Series(imp.importances_mean, index=FEATS).sort_values(ascending=False).round(2))

total_waiting    10.18
delta_1           9.57
delta_3           4.39
pct_gt18          3.85
pct_gt52          3.06
month_of_year     2.76
delta_2           2.22
occupancy_ga      0.20
dtype: float64


In [8]:
# Goal: store the validation forecasts so these numbers stay re-readable without a rerun.
out = val[["provider_code", "month"]].copy()
out["target"] = truth
out["pred_naive"] = val["total_waiting"].to_numpy()
out["pred_gbm_full"] = preds["full-history"]
out["pred_gbm_post"] = preds["post-covid"]
out.to_csv(PROC / "val_forecasts_c110_gbm.csv", index=False)
print(out.shape, "->", PROC / "val_forecasts_c110_gbm.csv")

(1407, 6) -> ..\data\processed\val_forecasts_c110_gbm.csv
